# Import everything

In [1]:
# works from base [Mark]
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.widgets import Slider
import matplotlib.gridspec as gridspec
from mpl_toolkits.mplot3d import Axes3D
import sys
import types
import os
from pathlib import Path
try:
    import _dropbox_wrapper as dbw
except ImportError:
    dbw = None

# To run the interactive plot in cell 3, install this:
# conda install -c conda-forge ipympl ipywidgets
%matplotlib inline
# Works in Jupyter Notebook version 7.0.8
from IPython.display import display, HTML
display(HTML("<style>.jp-Cell { margin-left: -20% !important; margin-right: -15% !important; }</style>"))

# Load recovery

In [2]:
recovery_filename = '../../../DATA/drum5_chirp_22000Hz_vol_30_10by10_08_10_Time_13_39_15/RECOVERY.npz'
# 0) Define your placeholder function exactly as pickle expects it
def compute_CAM2_translations_v3_cupy(*args, **kwargs):
    # either a no‑op or your real implementation
    return None

# 1) Inject (or patch) the recover_core_lib module *before* loading
if 'recover_core_lib' in sys.modules:
    # if it was already imported, just add the missing name
    sys.modules['recover_core_lib'].compute_CAM2_translations_v3_cupy = compute_CAM2_translations_v3_cupy
else:
    fake = types.ModuleType('recover_core_lib')
    fake.compute_CAM2_translations_v3_cupy = compute_CAM2_translations_v3_cupy
    sys.modules['recover_core_lib'] = fake

if not os.path.isfile(recovery_filename) and dbw is not None:
    recovery_data   = np.load(dbw.dbox_read_file(Path(recovery_filename).as_posix()), allow_pickle=True)
else:
    recovery_data   = np.load(recovery_filename, allow_pickle=True)

all_shifts        = recovery_data['all_shifts']
all_params        = recovery_data['all_params']
loaded_filename   = recovery_data['loaded_filename']
run_opt           = recovery_data['run_opt'].item()

print("Recovery file loaded successfully:")
print(" - File:", recovery_filename)
print(" - all_shifts shape:", all_shifts.shape)
print(" - all_params keys:", list(all_params.keys()) if isinstance(all_params, dict) else "Not a dict")
print(" - run_opt keys:", list(run_opt.keys()) if isinstance(run_opt, dict) else "Not a dict")
print(" - fs:", run_opt['cam_params']['camera_FPS'])
all_shifts=all_shifts[:,1:]

Recovery file loaded successfully:
 - File: ../../../DATA/drum5_chirp_22000Hz_vol_30_10by10_08_10_Time_13_39_15/RECOVERY.npz
 - all_shifts shape: (100, 374000, 2)
 - all_params keys: Not a dict
 - run_opt keys: ['cam_params', 'run_opt_multiROIs', 'run_opt_recovery', 'run_dict']
 - fs: 22000


# Filter the signals

In [3]:
from scipy.signal import butter, sosfiltfilt

def bandpass_filter(data, lowcut, highcut, fs, order=5):
    sos = butter(order, [lowcut, highcut], fs=fs, btype='band', output='sos')
    return sosfiltfilt(sos, data)

lowcut = 50
highcut = 10000
fs = run_opt['cam_params']['camera_FPS']
order = 5

if 1:
    filtered_shifts = np.empty_like(all_shifts)
    for i in range(all_shifts.shape[0]):
        for j in range(all_shifts.shape[2]):
            filtered_shifts[i, :, j] = bandpass_filter(all_shifts[i, :, j], lowcut, highcut, fs, order)
else:
    filtered_shifts = all_shifts

# Figure signals

In [4]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import LightSource
import os

# (Your reconstruct_surface_from_gradients function goes here, unchanged)
def reconstruct_surface_from_gradients(
    DX: np.ndarray,
    DY: np.ndarray,
    dx: float = 1.0,
    dy: float = 1.0,
    smoothing_lambda: float | None = None,
    smoothing_length: float | None = None,
    zero_mean: bool = True,
) -> np.ndarray:
    if DX.shape != DY.shape:
        raise ValueError("DX and DY must have the same shape (Nx, Ny).")
    if not (np.isrealobj(DX) and np.isrealobj(DY)):
        raise ValueError("DX and DY must be real arrays.")

    DX = np.asarray(DX, dtype=np.float64)
    DY = np.asarray(DY, dtype=np.float64)
    Nx, Ny = DX.shape

    # Angular spatial frequencies consistent with numpy FFTs
    kx = 2.0 * np.pi * np.fft.fftfreq(Nx, d=dx)[:, None]    # (Nx, 1)
    ky = 2.0 * np.pi * np.fft.fftfreq(Ny, d=dy)[None, :]    # (1, Ny)

    # FFT of gradients
    DX_hat = np.fft.fft2(DX)
    DY_hat = np.fft.fft2(DY)

    # Smoothing λ (screened Poisson)
    if smoothing_lambda is None:
        if smoothing_length is not None and smoothing_length > 0:
            lam = 2.0 * np.pi / float(smoothing_length)
        else:
            lam = 0.0
    else:
        lam = float(smoothing_lambda)
        if lam < 0:
            raise ValueError("smoothing_lambda must be ≥ 0")
            
    denom = (kx**2 + ky**2)

    if lam > 0.0:
        denom = denom + lam**2

    # Avoid division by zero at DC; numerator is 0 there for consistent gradients
    denom = denom.astype(np.float64)
    denom[0, 0] = np.inf

    # Frankot–Chellappa least-squares solution
    Z_hat = (-1j * kx * DX_hat - 1j * ky * DY_hat) / denom
    Z_hat[0, 0] = 0.0  # fix the arbitrary constant height
    Z = np.fft.ifft2(Z_hat).real
    if zero_mean:

        Z -= Z.mean()

    return Z

    
def compute_normalized_shading(Z, global_z_lims, ls, base_rgb):
    """
    Compute shading with intensity normalized by actual surface amplitude.
    (This function is unchanged)
    """
    # Calculate actual amplitude of this frame's surface
    z_range_current = Z.max() - Z.min()
    z_range_global = global_z_lims[1] - global_z_lims[0]
    
    # Amplitude ratio: 0 when flat, 1.0 when at maximum amplitude
    amplitude_ratio = z_range_current / (z_range_global + 1e-10)
    amplitude_ratio = np.clip(amplitude_ratio, 0, 1)

    # Apply power function to make shading more aggressive
    amplitude_ratio = amplitude_ratio ** 0.5

    # Create RGB array for the surface
    rgb = np.ones((*Z.T.shape, 3))
    rgb[..., 0] = base_rgb[0]
    rgb[..., 1] = base_rgb[1]
    rgb[..., 2] = base_rgb[2]
    
    # Compute shading
    shaded = ls.shade_rgb(rgb, Z.T)
    
    # Blend between uniform color and shaded based on amplitude
    uniform_color = np.ones_like(shaded) * base_rgb
    facecolors = uniform_color * (1 - amplitude_ratio) + shaded * amplitude_ratio
    
    return facecolors

#
# ==============================================================================
#  FUNCTION 1: ORIGINAL (WITH NEW BLUE COLOR)
# ==============================================================================
#
def generate_animation_frame(file_index, output_folder="animation_frames"):
    """
    Plots a SINGLE pre-calculated frame for the ORIGINAL animation.
    """
    
    # 1. --- Get pre-calculated data for this frame --
    U1 = U1_frames[file_index]
    V1 = V1_frames[file_index]
    Z1 = Z1_frames[file_index]
    
    U2 = U2_frames[file_index]
    V2 = V2_frames[file_index]
    Z2 = Z2_frames[file_index]
    
    t1_start = t1_starts[file_index]
    t1_end = t1_ends[file_index]
    t2_start = t2_starts[file_index]
    t2_end = t2_ends[file_index]

    # 2. --- Create the figure --
    fig = plt.figure(figsize=(18, 8), facecolor='black')
    
    gs = gridspec.GridSpec(2, 5, figure=fig, 
                            width_ratios=[1.5, 0.15, 1.5, 0.01, 1], 
                            height_ratios=[1, 1],
                            hspace=0.3,
                            wspace=0) 

    # ========== ROW 1: First frequency (f1) ==========
    
    # --- Plot 1.1: Time-domain trace (f1) ---
    gs00 = gridspec.GridSpecFromSubplotSpec(3, 1, subplot_spec=gs[0, 0], height_ratios=[0.125, 0.75, 0.125])
    ax31 = fig.add_subplot(gs00[1, 0], facecolor='black') 
    
    ax31.plot(time_axis[t1_start:t1_end]* 1000, kill[point_yellow[1], point_yellow[0], t1_start:t1_end,0], color='#ffff00', linestyle='-', linewidth=1.5)
    # *** MODIFICATION: Use hex color ***
    ax31.plot(time_axis[t1_start:t1_end]* 1000, kill[point_blue[1], point_blue[0], t1_start:t1_end,0], color='#21c5f0', linestyle='-', linewidth=1.5)
    ax31.set_title('198.12 Hz', color='white'); ax31.set_ylabel('x-axis\npixel shift', fontsize=11, color='white')
    ax31.tick_params(labelsize=10, colors='white'); ax31.locator_params(axis='x', nbins=5)
    ax31.text(0.96, 0.09, 'time [ms]', transform=ax31.transAxes, fontsize=11, ha='right', va='bottom', bbox=bbox_props, color='white')
    ax31.spines['left'].set_color('white'); ax31.spines['bottom'].set_color('white')
    ax31.spines['right'].set_color('white'); ax31.spines['top'].set_color('white')

    # --- Plot 1.2: Gradient (f1) ---
    ax11 = fig.add_subplot(gs[0, 2], facecolor='black') 
    ax11.quiver(x_coords, y_coords, U1, V1, scale=global_quiver_scale1, width=0.01, color='white')
    ax11.plot(point_yellow[1], point_yellow[0], 'o', color='#fff100', markersize=10, markerfacecolor='none', markeredgewidth=2)
    # *** MODIFICATION: Use hex color ***
    ax11.plot(point_blue[1], point_blue[0], 'o', color='#21c5f0', markersize=10, markerfacecolor='none', markeredgewidth=2)
    ax11.set_xlim(-0.5, n_points - 0.5); ax11.set_ylim(-0.5, n_points - 0.5)
    ax11.set_aspect('equal'); ax11.set_title('mode gradients at 198.12 Hz', color='white')
    ax11.tick_params(labelsize=10, colors='white')
    ax11.spines['left'].set_color('white'); ax11.spines['bottom'].set_color('white')
    ax11.spines['right'].set_color('white'); ax11.spines['top'].set_color('white')

    # --- Plot 1.3: 3D Reconstruction (f1) ---
    ax12 = fig.add_subplot(gs[0, 4], projection='3d', facecolor='black') 
    ax12.set_title('surface reconstruction', color='white')
    ls = LightSource(azdeg=315, altdeg=45)
    base_rgb1 = (70/255, 130/255, 180/255) # This is the surface color, not the plot element
    facecolors1 = compute_normalized_shading(Z1, global_z1_lims, ls, base_rgb1)
    
    surf1 = ax12.plot_surface(X_m, Y_m, Z1.T, facecolors=facecolors1, 
                              linewidth=0, antialiased=True, alpha=0.9, 
                              rstride=1, cstride=1)
    ax12.set_xticklabels([]); ax12.set_yticklabels([]); ax12.set_zticklabels([])
    ax12.set_zlim(global_z1_lims)
    ax12.view_init(elev=25, azim=-60)
    ax12.xaxis.pane.set_color((0.0, 0.0, 0.0, 0.0)); ax12.yaxis.pane.set_color((0.0, 0.0, 0.0, 0.0))
    ax12.zaxis.pane.set_color((0.0, 0.0, 0.0, 0.0)); ax12.grid(False)

    # ========== ROW 2: Second frequency (f2) ==========

    # --- Plot 2.1: Time-domain trace (f2) ---
    gs10 = gridspec.GridSpecFromSubplotSpec(3, 1, subplot_spec=gs[1, 0], height_ratios=[0.125, 0.75, 0.125])
    ax32 = fig.add_subplot(gs10[1, 0], facecolor='black') 
    
    ax32.plot(time_axis[t2_start:t2_end]* 1000, kill[point_yellow[1], point_yellow[0], t2_start:t2_end,0],  color='#ffff00', linestyle='-', linewidth=1.5)
    # *** MODIFICATION: Use hex color ***
    ax32.plot(time_axis[t2_start:t2_end]* 1000, kill[point_blue[1], point_blue[0], t2_start:t2_end,0], color='#21c5f0', linestyle='-', linewidth=1.5)
    ax32.set_title('411.35 Hz', color='white'); ax32.tick_params(labelsize=10, colors='white')
    ax32.set_ylabel('x-axis\npixel shift', fontsize=11, color='white') 
    ax32.locator_params(axis='x', nbins=5)
    ax32.text(0.96, 0.09, 'time [ms]', transform=ax32.transAxes, fontsize=11, ha='right', va='bottom', bbox=bbox_props, color='white')
    ax32.spines['left'].set_color('white'); ax32.spines['bottom'].set_color('white')
    ax32.spines['right'].set_color('white'); ax32.spines['top'].set_color('white')

    # --- Plot 2.2: Gradient (f2) ---
    ax21 = fig.add_subplot(gs[1, 2], facecolor='black') 
    ax21.quiver(x_coords, y_coords, U2, V2, scale=global_quiver_scale2, width=0.01, color='white')
    ax21.plot(point_yellow[1], point_yellow[0], 'o', color='#fff100', markersize=10, markerfacecolor='none', markeredgewidth=2)
    # *** MODIFICATION: Use hex color ***
    ax21.plot(point_blue[1], point_blue[0], 'o', color='#21c5f0', markersize=10, markerfacecolor='none', markeredgewidth=2)
    ax21.set_xlim(-0.5, n_points - 0.5); ax21.set_ylim(-0.5, n_points - 0.5)
    ax21.set_aspect('equal'); ax21.set_title('mode gradients at 411.35 Hz', color='white')
    ax21.tick_params(labelsize=10, colors='white')
    ax21.spines['left'].set_color('white'); ax21.spines['bottom'].set_color('white')
    ax21.spines['right'].set_color('white'); ax21.spines['top'].set_color('white')

    # --- Plot 2.3: 3D Reconstruction (f2) ---
    ax22 = fig.add_subplot(gs[1, 4], projection='3d', facecolor='black') 
    ax22.set_title('surface reconstruction', color='white')
    base_rgb2 = (70/255, 130/255, 180/255) # Surface color
    facecolors2 = compute_normalized_shading(Z2, global_z2_lims, ls, base_rgb2)
    
    surf2 = ax22.plot_surface(X_m, Y_m, Z2.T, facecolors=facecolors2, 
                              linewidth=0, antialiased=True, alpha=0.9, 
                              rstride=1, cstride=1)
    ax22.set_xticklabels([]); ax22.set_yticklabels([]); ax22.set_zticklabels([])
    ax22.set_zlim(global_z2_lims)
    ax22.view_init(elev=25, azim=-150)
    ax22.xaxis.pane.set_color((0.0, 0.0, 0.0, 0.0)); ax22.yaxis.pane.set_color((0.0, 0.0, 0.0, 0.0))
    ax22.zaxis.pane.set_color((0.0, 0.0, 0.0, 0.0)); ax22.grid(False)

    # 4. --- Standardize layout and Save/Close Figure ---
    filename_png = os.path.join(output_folder, f"frame_{file_index:04d}.png")
    plt.savefig(filename_png, dpi=300, facecolor=fig.get_facecolor())
    plt.close(fig)
    
    return True

#
# ==============================================================================
#  FUNCTION 2: NEW SUPERPOSITION PLOT (WITH NEW Y-LIMITS)
# ==============================================================================
#
def generate_animation_frame_superposition(file_index, output_folder="animation_frames_superposition"):
    """
    Plots a SINGLE pre-calculated frame for the NEW SUPERPOSITION animation.
    *** MODIFIED: This is a 2-row plot. ***
    *** The time-domain plots (ax31, ax32) show the SUM of yellow and blue points ***
    *** in a single green line, with a minimum y-axis of +/- 0.3 ***
    """
    
    # 1. --- Get pre-calculated data for this frame --
    U1 = U1_frames[file_index]
    V1 = V1_frames[file_index]
    Z1 = Z1_frames[file_index]
    
    U2 = U2_frames[file_index]
    V2 = V2_frames[file_index]
    Z2 = Z2_frames[file_index]
    
    t1_start = t1_starts[file_index]
    t1_end = t1_ends[file_index]
    t2_start = t2_starts[file_index]
    t2_end = t2_ends[file_index]

    # 2. --- Create the figure (Identical layout to Function 1) --
    fig = plt.figure(figsize=(18, 8), facecolor='black')
    
    gs = gridspec.GridSpec(2, 5, figure=fig, 
                            width_ratios=[1.5, 0.15, 1.5, 0.01, 1], 
                            height_ratios=[1, 1],
                            hspace=0.3,
                            wspace=0) 

    # ========== ROW 1: First frequency (f1) ==========
    
    # --- Plot 1.1: Time-domain trace (f1) ---
    # *** THIS PLOT IS THE PRIMARY MODIFICATION ***
    gs00 = gridspec.GridSpecFromSubplotSpec(3, 1, subplot_spec=gs[0, 0], height_ratios=[0.125, 0.75, 0.125])
    ax31 = fig.add_subplot(gs00[1, 0], facecolor='black') 
    
    # Calculate the signals at the two points
    signal_yellow_1 = kill[point_yellow[1], point_yellow[0], t1_start:t1_end, 0]
    signal_blue_1 = kill[point_blue[1], point_blue[0], t1_start:t1_end, 0]
    
    # Sum them to create the new green signal
    signal_green_1 = signal_yellow_1 + signal_blue_1
    
    # Plot the single green line
    ax31.plot(time_axis[t1_start:t1_end]* 1000, signal_green_1, color='#00ff00', linestyle='-', linewidth=1.5) 
    
    # Apply new Y-axis limit logic
    min_val = np.min(signal_green_1)
    max_val = np.max(signal_green_1)
    # Add 10% padding, then enforce the 0.3 limit
    # *** MODIFICATION: Use 0.3 ***
    plot_min = min(-0.3, min_val - 0.1 * abs(min_val))
    plot_max = max( 0.3, max_val + 0.1 * abs(max_val))
    ax31.set_ylim(plot_min, plot_max)
    
    # Standard plot setup
    ax31.set_title('superposed signal at 198.12 Hz', color='white'); ax31.set_ylabel('x-axis\npixel shift', fontsize=11, color='white')
    ax31.tick_params(labelsize=10, colors='white'); ax31.locator_params(axis='x', nbins=5)
    ax31.text(0.96, 0.09, 'time [ms]', transform=ax31.transAxes, fontsize=11, ha='right', va='bottom', bbox=bbox_props, color='white')
    ax31.spines['left'].set_color('white'); ax31.spines['bottom'].set_color('white')
    ax31.spines['right'].set_color('white'); ax31.spines['top'].set_color('white')

    # --- Plot 1.2: Gradient (f1) ---
    # (This plot is identical to Function 1, including new blue color)
    ax11 = fig.add_subplot(gs[0, 2], facecolor='black') 
    ax11.quiver(x_coords, y_coords, U1, V1, scale=global_quiver_scale1, width=0.01, color='white')
    ax11.plot(point_yellow[1], point_yellow[0], 'o', color='#fff100', markersize=10, markerfacecolor='none', markeredgewidth=2)
    # *** MODIFICATION: Use hex color ***
    ax11.plot(point_blue[1], point_blue[0], 'o', color='#21c5f0', markersize=10, markerfacecolor='none', markeredgewidth=2)
    ax11.set_xlim(-0.5, n_points - 0.5); ax11.set_ylim(-0.5, n_points - 0.5)
    ax11.set_aspect('equal'); ax11.set_title('mode gradients at 198.12 Hz', color='white')
    ax11.tick_params(labelsize=10, colors='white')
    ax11.spines['left'].set_color('white'); ax11.spines['bottom'].set_color('white')
    ax11.spines['right'].set_color('white'); ax11.spines['top'].set_color('white')

    # --- Plot 1.3: 3D Reconstruction (f1) ---
    # (This plot is identical to Function 1)
    ax12 = fig.add_subplot(gs[0, 4], projection='3d', facecolor='black') 
    ax12.set_title('surface reconstruction', color='white')
    ls = LightSource(azdeg=315, altdeg=45)
    base_rgb1 = (70/255, 130/255, 180/255) 
    facecolors1 = compute_normalized_shading(Z1, global_z1_lims, ls, base_rgb1)
    
    surf1 = ax12.plot_surface(X_m, Y_m, Z1.T, facecolors=facecolors1, 
                              linewidth=0, antialiased=True, alpha=0.9, 
                              rstride=1, cstride=1)
    ax12.set_xticklabels([]); ax12.set_yticklabels([]); ax12.set_zticklabels([])
    ax12.set_zlim(global_z1_lims)
    ax12.view_init(elev=25, azim=-60)
    ax12.xaxis.pane.set_color((0.0, 0.0, 0.0, 0.0)); ax12.yaxis.pane.set_color((0.0, 0.0, 0.0, 0.0))
    ax12.zaxis.pane.set_color((0.0, 0.0, 0.0, 0.0)); ax12.grid(False)

    # ========== ROW 2: Second frequency (f2) ==========

    # --- Plot 2.1: Time-domain trace (f2) ---
    # *** THIS PLOT IS THE SECONDARY MODIFICATION ***
    gs10 = gridspec.GridSpecFromSubplotSpec(3, 1, subplot_spec=gs[1, 0], height_ratios=[0.125, 0.75, 0.125])
    ax32 = fig.add_subplot(gs10[1, 0], facecolor='black') 
    
    # Calculate the signals at the two points for the f2 window
    signal_yellow_2 = kill[point_yellow[1], point_yellow[0], t2_start:t2_end, 0]
    signal_blue_2 = kill[point_blue[1], point_blue[0], t2_start:t2_end, 0]
    
    # Sum them to create the new green signal
    signal_green_2 = signal_yellow_2 + signal_blue_2
    
    # Plot the single green line
    ax32.plot(time_axis[t2_start:t2_end]* 1000, signal_green_2, color='#00ff00', linestyle='-', linewidth=1.5) 
    
    # Apply new Y-axis limit logic
    min_val = np.min(signal_green_2)
    max_val = np.max(signal_green_2)
    # Add 10% padding, then enforce the 0.3 limit
    # *** MODIFICATION: Use 0.3 ***
    plot_min = min(-0.3, min_val - 0.1 * abs(min_val))
    plot_max = max( 0.3, max_val + 0.1 * abs(max_val))
    ax32.set_ylim(plot_min, plot_max)
    
    # Standard plot setup
    ax32.set_title('superposed signal at 411.35 Hz', color='white'); ax32.tick_params(labelsize=10, colors='white')
    ax32.set_ylabel('x-axis\npixel shift', fontsize=11, color='white') 
    ax32.locator_params(axis='x', nbins=5)
    ax32.text(0.96, 0.09, 'time [ms]', transform=ax32.transAxes, fontsize=11, ha='right', va='bottom', bbox=bbox_props, color='white')
    ax32.spines['left'].set_color('white'); ax32.spines['bottom'].set_color('white')
    ax32.spines['right'].set_color('white'); ax32.spines['top'].set_color('white')

    # --- Plot 2.2: Gradient (f2) ---
    # (This plot is identical to Function 1, including new blue color)
    ax21 = fig.add_subplot(gs[1, 2], facecolor='black') 
    ax21.quiver(x_coords, y_coords, U2, V2, scale=global_quiver_scale2, width=0.01, color='white')
    ax21.plot(point_yellow[1], point_yellow[0], 'o', color='#fff100', markersize=10, markerfacecolor='none', markeredgewidth=2)
    # *** MODIFICATION: Use hex color ***
    ax21.plot(point_blue[1], point_blue[0], 'o', color='#21c5f0', markersize=10, markerfacecolor='none', markeredgewidth=2)
    ax21.set_xlim(-0.5, n_points - 0.5); ax21.set_ylim(-0.5, n_points - 0.5)
    ax21.set_aspect('equal'); ax21.set_title('mode gradients at 411.35 Hz', color='white')
    ax21.tick_params(labelsize=10, colors='white')
    ax21.spines['left'].set_color('white'); ax21.spines['bottom'].set_color('white')
    ax21.spines['right'].set_color('white'); ax21.spines['top'].set_color('white')

    # --- Plot 2.3: 3D Reconstruction (f2) ---
    # (This plot is identical to Function 1)
    ax22 = fig.add_subplot(gs[1, 4], projection='3d', facecolor='black') 
    ax22.set_title('surface reconstruction', color='white')
    base_rgb2 = (70/255, 130/255, 180/255) 
    facecolors2 = compute_normalized_shading(Z2, global_z2_lims, ls, base_rgb2)
    
    surf2 = ax22.plot_surface(X_m, Y_m, Z2.T, facecolors=facecolors2, 
                              linewidth=0, antialiased=True, alpha=0.9, 
                              rstride=1, cstride=1)
    ax22.set_xticklabels([]); ax22.set_yticklabels([]); ax22.set_zticklabels([])
    ax22.set_zlim(global_z2_lims)
    ax22.view_init(elev=25, azim=-150)
    ax22.xaxis.pane.set_color((0.0, 0.0, 0.0, 0.0)); ax22.yaxis.pane.set_color((0.0, 0.0, 0.0, 0.0))
    ax22.zaxis.pane.set_color((0.0, 0.0, 0.0, 0.0)); ax22.grid(False)

    # 4. --- Standardize layout and Save/Close Figure ---
    # *** MODIFICATION: Save to a unique filename pattern ***
    filename_png = os.path.join(output_folder, f"frame_super_{file_index:04d}.png")
    
    plt.savefig(filename_png, dpi=300, facecolor=fig.get_facecolor())
    
    plt.close(fig)
    
    return True

In [5]:
import shutil # For deleting the old folder
import time

# =========================================================
# ========== 1. DATA & ANIMATION SETUP ====================
# =========================================================
print("Setting up global variables for animation...")
start_setup_time = time.time()

# --- Part A: Variables from your original script ---
# (ASSUMES all_shifts, filtered_shifts, run_opt ARE ALREADY LOADED)

start_idx1 = 71148
start_idx2 = 131704

n_points = 10
D = 0.01
show_shifts = filtered_shifts 

# --- Part B: My setup logic (now using the variables above) ---

# 1. Base indices
base_start_idx1 = start_idx1
base_start_idx2 = start_idx2

# 2. Points
point_yellow = (2, 2)
# *** MODIFICATION: Renamed 'point_green' to 'point_blue' ***
point_blue = (7, 6)

# 3. Pre-calculated variables
fs = run_opt['cam_params']['camera_FPS']
n_pts = all_shifts.shape[1]
time_axis = np.arange(n_pts) / fs
x_coords, y_coords = np.meshgrid(np.arange(n_points), np.arange(n_points))
X_m = x_coords * D
Y_m = y_coords * D

# 4. Reshaped data
kill = show_shifts.reshape(10,10,n_pts,2)

# 5. Linear indices
idx_yellow_linear = point_yellow[0] * n_points + point_yellow[1]
# *** MODIFICATION: Use 'point_blue' ***
idx_blue_linear = point_blue[0] * n_points + point_blue[1]

# 6. Plot properties
bbox_props = dict(boxstyle="round,pad=0.5",
                  fc="#1A1A1A", # Dark gray background
                  ec="none",
                  alpha=0.7)

# --- Part C: Pre-calculate all frames and Z-Limits (THE FIX) ---
print("Pre-calculating all frames for animation (this may take a moment)...")

# Animation parameters (set them here)
start_frame_calc = -150
num_frames_calc = 2400

# Create empty lists to store all frame data
U1_frames, V1_frames, Z1_frames = [], [], []
U2_frames, V2_frames, Z2_frames = [], [], []
t1_starts, t1_ends, t2_starts, t2_ends = [], [], [], []
# (No superposition lists needed here)

global_z1_min, global_z1_max = np.inf, -np.inf
global_z2_min, global_z2_max = np.inf, -np.inf
# (No superposition z-limits needed here)

# Add new trackers for global quiver magnitude
global_mag1_max = -np.inf
global_mag2_max = -np.inf
# (No superposition quiver limits needed here)


for i, frame_num in enumerate(range(start_frame_calc, start_frame_calc + num_frames_calc)):
    
    current_idx1 = base_start_idx1 + frame_num
    current_idx2 = base_start_idx2 + frame_num
    
    t_window_size = 300
    t1_start = current_idx1
    t1_end = t1_start + t_window_size
    t2_start = current_idx2
    t2_end = t2_start + t_window_size
    
    # Safety check
    if t1_start < 0 or t2_start < 0 or t1_end >= n_pts or t2_end >= n_pts:
        print(f"Warning: Frame {frame_num} is out of data bounds. Stopping pre-calculation.")
        break # Stop if we hit the end of the data
    
    if i % 20 == 0: # Print progress
        print(f"  Calculating frame {i+1} / {num_frames_calc} (t={frame_num})...")

    # --- Calculate all data for this frame ---
    U1 = kill[..., current_idx1, 0]
    V1 = kill[..., current_idx1, 1]
    U2 = kill[..., current_idx2, 0]
    V2 = kill[..., current_idx2, 1]

    Z1 = reconstruct_surface_from_gradients(U1.T, V1.T, dx=D, dy=D, smoothing_length=0.1, zero_mean=True)
    Z2 = reconstruct_surface_from_gradients(U2.T, V2.T, dx=D, dy=D, smoothing_length=0.1, zero_mean=True)

    mag1 = np.sqrt(U1**2 + V1**2)
    mag2 = np.sqrt(U2**2 + V2**2)

    # Update global max magnitudes
    if np.max(mag1) > global_mag1_max: global_mag1_max = np.max(mag1)
    if np.max(mag2) > global_mag2_max: global_mag2_max = np.max(mag2)

    # --- Store everything in global lists --
    U1_frames.append(U1); V1_frames.append(V1); Z1_frames.append(Z1); 
    U2_frames.append(U2); V2_frames.append(V2); Z2_frames.append(Z2);
    t1_starts.append(t1_start); t1_ends.append(t1_end)
    t2_starts.append(t2_start); t2_ends.append(t2_end)
    
    # --- Update global Z-limits ---
    if Z1.min() < global_z1_min: global_z1_min = Z1.min()
    if Z1.max() > global_z1_max: global_z1_max = Z1.max()
    if Z2.min() < global_z2_min: global_z2_min = Z2.min()
    if Z2.max() > global_z2_max: global_z2_max = Z2.max()
    
    # (No superposition calculation needed)

# --- Finalize global quiver scales ---
global_quiver_scale1 = global_mag1_max / 0.13 if global_mag1_max > 0 else 1.0
global_quiver_scale2 = global_mag2_max / 0.13 if global_mag2_max > 0 else 1.0
print(f"Global Quiver Scale 1 set to: {global_quiver_scale1}")
print(f"Global Quiver Scale 2 set to: {global_quiver_scale2}")
        
# --- Finalize global limits ---
z1_abs_max = max(np.abs(global_z1_min), np.abs(global_z1_max)) * 2.0
z2_abs_max = max(np.abs(global_z2_min), np.abs(global_z2_max)) * 2.0

if z1_abs_max == 0.0: z1_abs_max = 1.0 
if z2_abs_max == 0.0: z2_abs_max = 1.0

global_z1_lims = [-z1_abs_max, z1_abs_max]
global_z2_lims = [-z2_abs_max, z2_abs_max]

num_frames_calculated = len(Z1_frames)
print(f"Pre-calculation complete. {num_frames_calculated} frames processed.")
print(f"Global Z1 limits set to: {global_z1_lims}")
print(f"Global Z2 limits set to: {global_z2_lims}")


# 7. Clean out old frames
output_folder = "animation_frames"
# *** NEW: Define second output folder ***
output_folder_super = "animation_frames_superposition"

# *** NEW: Loop over both folders to create/clear them ***
for folder in [output_folder, output_folder_super]:
    if os.path.isdir(folder):
        print(f"Deleting old frames from '{folder}'...")
        shutil.rmtree(folder)
    os.makedirs(folder, exist_ok=True)

end_setup_time = time.time()
print(f"Setup complete in {end_setup_time - start_setup_time:.2f} seconds.")
print(f"Frames will be saved to: {os.path.abspath(output_folder)}")
# *** NEW: Print path for new folder ***
print(f"Superposition frames will be saved to: {os.path.abspath(output_folder_super)}")

Setting up global variables for animation...
Pre-calculating all frames for animation (this may take a moment)...
  Calculating frame 1 / 2400 (t=-150)...
  Calculating frame 21 / 2400 (t=-130)...
  Calculating frame 41 / 2400 (t=-110)...
  Calculating frame 61 / 2400 (t=-90)...
  Calculating frame 81 / 2400 (t=-70)...
  Calculating frame 101 / 2400 (t=-50)...
  Calculating frame 121 / 2400 (t=-30)...
  Calculating frame 141 / 2400 (t=-10)...
  Calculating frame 161 / 2400 (t=10)...
  Calculating frame 181 / 2400 (t=30)...
  Calculating frame 201 / 2400 (t=50)...
  Calculating frame 221 / 2400 (t=70)...
  Calculating frame 241 / 2400 (t=90)...
  Calculating frame 261 / 2400 (t=110)...
  Calculating frame 281 / 2400 (t=130)...
  Calculating frame 301 / 2400 (t=150)...
  Calculating frame 321 / 2400 (t=170)...
  Calculating frame 341 / 2400 (t=190)...
  Calculating frame 361 / 2400 (t=210)...
  Calculating frame 381 / 2400 (t=230)...
  Calculating frame 401 / 2400 (t=250)...
  Calculatin

## Animation loop

In [6]:
# =========================================================
# ========== 🎬 ANIMATION LOOP 1 (Original) ================
# =========================================================
output_folder = "animation_frames"
start_plot_time = time.time()

print("Starting animation loop (Original: f1, f2)...")

# Use the number of frames we *actually* calculated
total_frames_to_plot = len(Z1_frames) 

for file_index in range(total_frames_to_plot):
    
    if file_index % 20 == 0: # Print progress
        print(f"  Plotting frame {file_index+1} / {total_frames_to_plot}...")
    
    # Call the ORIGINAL (blue-modified) function
    success = generate_animation_frame(file_index, output_folder)
    
    if not success:
        print(f"Error plotting frame {file_index}, stopping.")
        break

end_plot_time = time.time()
print("---")
print(f"✅ Original plotting complete. Generated {total_frames_to_plot} frames in {end_plot_time - start_plot_time:.2f} seconds.")


# =========================================================
# ========== 🎬 ANIMATION LOOP 2 (Superposition) ==========
# =========================================================
output_folder_super = "animation_frames_superposition"
start_plot_time_super = time.time()

print("\nStarting animation loop (Superposition: f1 + f2)...")

# total_frames_to_plot is the same
for file_index in range(total_frames_to_plot):
    
    if file_index % 20 == 0: # Print progress
        print(f"  Plotting superposition frame {file_index+1} / {total_frames_to_plot}...")
    
    # Call the NEW SUPERPOSITION function
    success = generate_animation_frame_superposition(file_index, output_folder_super)
    
    if not success:
        print(f"Error plotting frame {file_index}, stopping.")
        break

end_plot_time_super = time.time()
print("---")
print(f"✅ Superposition plotting complete. Generated {total_frames_to_plot} frames in {end_plot_time_super - start_plot_time_super:.2f} seconds.")

Starting animation loop (Original: f1, f2)...
  Plotting frame 1 / 2400...
  Plotting frame 21 / 2400...
  Plotting frame 41 / 2400...
  Plotting frame 61 / 2400...
  Plotting frame 81 / 2400...
  Plotting frame 101 / 2400...
  Plotting frame 121 / 2400...
  Plotting frame 141 / 2400...
  Plotting frame 161 / 2400...
  Plotting frame 181 / 2400...
  Plotting frame 201 / 2400...
  Plotting frame 221 / 2400...
  Plotting frame 241 / 2400...
  Plotting frame 261 / 2400...
  Plotting frame 281 / 2400...
  Plotting frame 301 / 2400...
  Plotting frame 321 / 2400...
  Plotting frame 341 / 2400...
  Plotting frame 361 / 2400...
  Plotting frame 381 / 2400...
  Plotting frame 401 / 2400...
  Plotting frame 421 / 2400...
  Plotting frame 441 / 2400...
  Plotting frame 461 / 2400...
  Plotting frame 481 / 2400...
  Plotting frame 501 / 2400...
  Plotting frame 521 / 2400...
  Plotting frame 541 / 2400...
  Plotting frame 561 / 2400...
  Plotting frame 581 / 2400...
  Plotting frame 601 / 2400...

## Generate video

In [7]:
import glob
import os
from moviepy.editor import ImageSequenceClip

# ===============================================
# ========== 🎬 VIDEO 1 (Original) ==============
# ===============================================

output_folder = "animation_frames"
framerate = 60
output_video = "animation.mp4"

# Find all the .png files in the folder
file_list = sorted(glob.glob(os.path.join(output_folder, "frame_*.png")))
clip = None # Initialize clip outside the try block

if not file_list:
    print(f"Error: No .png frames found in folder '{output_folder}'.")
else:
    print(f"\n--- Creating Video 1 ---")
    print(f"Found {len(file_list)} frames. Creating video: {output_video}")
    
    try:
        # Create the video clip from the image sequence
        clip = ImageSequenceClip(file_list, fps=framerate)
        
        # Write the video file to disk
        clip.write_videofile(output_video, codec='libx264', audio=False, logger='bar')
        
        print(f"✅ Video 1 saved as: {output_video}")
        
    except Exception as e:
        print(f"An error occurred during video 1 creation: {e}")
        
    finally:
        if clip:
            clip.close()
            print("MoviePy clip 1 closed successfully.")

# ========================================================
# ========== 🎬 VIDEO 2 (Superposition) =================
# ========================================================

output_folder = "animation_frames_superposition" # <-- Use new folder
framerate = 60
output_video = "animation_superposition.mp4" # <-- Use new video name

# Find all the .png files in the folder
# *** MODIFICATION: Use 'frame_super_*.png' pattern ***
file_list = sorted(glob.glob(os.path.join(output_folder, "frame_super_*.png")))
clip = None # Re-initialize clip

if not file_list:
    print(f"Error: No .png frames found in folder '{output_folder}'.")
else:
    print(f"\n--- Creating Video 2 ---")
    print(f"Found {len(file_list)} frames. Creating video: {output_video}")
    
    try:
        # Create the video clip from the image sequence
        clip = ImageSequenceClip(file_list, fps=framerate)
        
        # Write the video file to disk
        clip.write_videofile(output_video, codec='libx264', audio=False, logger='bar')
        
        print(f"✅ Video 2 saved as: {output_video}")
        
    except Exception as e:
        print(f"An error occurred during video 2 creation: {e}")
        
    finally:
        if clip:
            clip.close()
            print("MoviePy clip 2 closed successfully.")


--- Creating Video 1 ---
Found 2400 frames. Creating video: animation.mp4
Moviepy - Building video animation.mp4.
Moviepy - Writing video animation.mp4



Moviepy - Done !
Moviepy - video ready animation.mp4
✅ Video 1 saved as: animation.mp4
MoviePy clip 1 closed successfully.

--- Creating Video 2 ---
Found 2400 frames. Creating video: animation_superposition.mp4
Moviepy - Building video animation_superposition.mp4.
Moviepy - Writing video animation_superposition.mp4



Moviepy - Done !
Moviepy - video ready animation_superposition.mp4
✅ Video 2 saved as: animation_superposition.mp4
MoviePy clip 2 closed successfully.
